# EDA — Tags & Labels

Exploratory analysis of the tag and label feature matrices, and the design decisions behind the
blended `album_genre_matrix`. The notebook follows the path the data took through the pipeline:

1. **The raw material** — what signal each source (album tags, artist tags, labels) actually carries
2. **The label problem** — why most label tags are genre noise, and how the allowlist isolates the useful ones
3. **Building the blend** — the three-tier strategy: universal album + artist tags, masked label reinforcement, allowlist label rescue
4. **Coverage vs richness** — the key distinction that explains why some changes help quality without moving the coverage number
5. **Column vocabulary** — what the frequency threshold prunes

**A note on two words used throughout:**
- **Coverage** — does an album have *any* genre signal at all? (a binary, per-album question)
- **Richness** — how many distinct genre tags does an already-covered album carry? (signal quality)

These are different axes. A change can add richness without adding coverage, and the analysis below
shows exactly that happening.

**Inputs:** `data/features/album_ids.pkl`, `data/features/artist_ids.pkl`,
`data/features/album_genre_matrix.npz`, plus raw parquets:
`mb_album_tag.parquet`, `mb_artist_tag.parquet`, `mb_album_label.parquet`, `mb_album_artists.parquet`.

All parquets are loaded once in the setup cell and reused throughout. No archived or intermediate
NPZ files are required — all per-source statistics are derived directly from the raw data.

**Run after:** `3-features/02-feature-genre.ipynb`. Display-only — writes nothing to `data/`.

## Setup

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import load_npz, csr_matrix

FEATURES_DIR         = '../data/features'
DATA_DIR             = '../data'
MIN_TAG_OCC          = 10
W_LABEL              = 0.3
LABEL_OVERLAP_THRESHOLD = 0.6

In [ ]:
# ── Index files ────────────────────────────────────────────────────────────
with open(f'{FEATURES_DIR}/album_ids.pkl', 'rb') as f:
    album_ids = pickle.load(f)
album_index = pd.Index(album_ids)
n_albums = len(album_index)

with open(f'{FEATURES_DIR}/artist_ids.pkl', 'rb') as f:
    artist_ids = pickle.load(f)
artist_index = pd.Index(artist_ids)

# ── Raw parquets (loaded once; reused by all downstream cells) ─────────────
album_tags_raw  = pd.read_parquet(f'{DATA_DIR}/mb_album_tag.parquet')
album_tags_raw  = album_tags_raw[album_tags_raw['tag_count'] > 0]

artist_tags_raw = pd.read_parquet(f'{DATA_DIR}/mb_artist_tag.parquet')
artist_tags_raw = artist_tags_raw[artist_tags_raw['tag_count'] > 0]

album_label_raw = pd.read_parquet(f'{DATA_DIR}/mb_album_label.parquet')

album_artists = (
    pd.read_parquet(f'{DATA_DIR}/mb_album_artists.parquet', columns=['album_id', 'artist_id'])
    .drop_duplicates(subset='album_id')
)

# Artist tags joined to albums (used by masking analysis and coverage comparison)
artist_tags_on_albums = (
    album_artists
    .merge(artist_tags_raw, on='artist_id', how='inner')
    [['album_id', 'tag_id', 'tag_count']]
)

# ── Unified genre vocabulary ────────────────────────────────────────────────
label_tags_for_vocab = (
    album_label_raw[album_label_raw['tag_count'] > 0][['tag_id', 'tag_count']]
)
_all_tags = pd.concat([
    album_tags_raw[['tag_id', 'tag_count']],
    artist_tags_on_albums[['tag_id', 'tag_count']],
    label_tags_for_vocab,
], ignore_index=True)
_tag_occ = _all_tags.groupby('tag_id')['tag_count'].sum()
genre_tag_index = pd.Index(sorted(_tag_occ[_tag_occ >= MIN_TAG_OCC].index))
n_tags = len(genre_tag_index)

# ── Per-row stats (no NPZ needed — derived from raw parquets) ──────────────
tags_per_album = (
    album_tags_raw.groupby('album_id')['tag_id'].nunique()
    .reindex(album_index, fill_value=0).values.astype(int)
)
labels_per_album = (
    album_label_raw.drop_duplicates(['album_id', 'label_id'])
    .groupby('album_id')['label_id'].nunique()
    .reindex(album_index, fill_value=0).values.astype(int)
)
tags_per_artist = (
    artist_tags_raw.groupby('artist_id')['tag_id'].nunique()
    .reindex(artist_index, fill_value=0).values.astype(int)
)

X_genre = load_npz(f'{FEATURES_DIR}/album_genre_matrix.npz')
genre_per_album = np.diff(X_genre.indptr)

# ── Column popularity (for structural plots and pruning analysis) ──────────
album_tag_popularity = (
    album_tags_raw.groupby('tag_id')['album_id'].nunique()
    .sort_values(ascending=False).values
)
album_label_popularity = (
    album_label_raw.drop_duplicates(['album_id', 'label_id'])
    .groupby('label_id')['album_id'].nunique()
    .sort_values(ascending=False).values
)
artist_tag_popularity = (
    artist_tags_raw.groupby('tag_id')['artist_id'].nunique()
    .sort_values(ascending=False).values
)
# Unsorted versions for threshold sweep in column-pruning cell
tag_col_counts   = album_tags_raw.groupby('tag_id')['album_id'].nunique().values
label_col_counts = (
    album_label_raw.drop_duplicates(['album_id', 'label_id'])
    .groupby('label_id')['album_id'].nunique().values
)

# ── Album tags mapped into genre vocab space (binary; reused by masking) ───
_at = album_tags_raw[album_tags_raw['tag_id'].isin(genre_tag_index)]
_at_row = album_index.get_indexer(_at['album_id'].values)
_at_col = genre_tag_index.get_indexer(_at['tag_id'].values)
_at_ok  = (_at_row >= 0) & (_at_col >= 0)
X_album_in_genre_vocab = csr_matrix(
    (np.ones(_at_ok.sum(), dtype='float32'), (_at_row[_at_ok], _at_col[_at_ok])),
    shape=(n_albums, n_tags),
)

print(f'album_ids        : {n_albums:,}')
print(f'artist_ids       : {len(artist_index):,}')
print(f'genre vocabulary : {n_tags:,} tags  (MIN_TAG_OCC={MIN_TAG_OCC})')
print(f'album_genre_matrix: {X_genre.shape}')

# Part 1 — The Raw Material

Three independent tag sources feed the genre blend. Before combining them, we look at how much
signal each one carries on its own and what shape that signal takes.

## Coverage by source

Every matrix is anchored to the full album/artist universe, so untagged entities exist as zero-rows.
The question here is how many rows are *not* zero — i.e. how much of the universe each source can
speak to at all.

In [ ]:
# All per-row stats pre-computed in load cell from raw parquets
def coverage(arr, label):
    n = len(arr)
    has = (arr > 0).sum()
    print(f'{label:<35} {has:>10,} / {n:>10,}  ({has/n*100:.1f}%)')

print(f"{'Source':<35} {'With signal':>10}   {'Total':>10}   Pct")
print('-' * 65)
coverage(tags_per_album,   'Album tags (direct)')
coverage(tags_per_artist,  'Artist tags')
coverage(labels_per_album, 'Album labels')
coverage(genre_per_album,  'Genre matrix (blended)')

### Each source alone is sparse; the blend is what reaches two-thirds of the universe

- **Album tags reach ~46%** — the baseline before any blending. Community tagging concentrates on
  popular releases; the other 54% are untagged.
- **Artist tags are the sparsest source (~9% of artists)** — heavily skewed to well-known acts.
- **Labels reach ~33%** — limited by whether commercial distribution data was entered.
- **The blended genre matrix reaches ~68%** — well above any single source, because the three fill
  different gaps.

## The shape of each source: profile complexity and the long tail

Two diagnostics per matrix:
- **Profile complexity (left):** non-zero entries per row — how many features a typical entity has.
- **Long-tail popularity (right):** column nnz, sorted, on a log scale — the power-law that governs
  which tags carry broad vs niche signal, and where the frequency prune bites.

In [ ]:
sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(3, 2, figsize=(16, 18))

# --- Album Tags ---
sns.histplot(tags_per_album, bins=range(0, 35), ax=axes[0, 0], color='#4A90E2', kde=True)
axes[0, 0].set_title('Album Tags: Profile Complexity (Tags per Album)', fontsize=12, weight='bold')
axes[0, 0].set_xlabel('Number of Unique Tags on a Single Album')
axes[0, 0].set_ylabel('Count of Albums')
axes[0, 0].set_xlim(0, 30)

# album_tag_popularity pre-computed in load cell (albums per tag, sorted desc)
axes[0, 1].plot(album_tag_popularity, color='#4A90E2', linewidth=2.5)
axes[0, 1].fill_between(range(len(album_tag_popularity)), album_tag_popularity, color='#4A90E2', alpha=0.25)
axes[0, 1].set_yscale('log')
axes[0, 1].set_title('Album Tags: Long-Tail Feature Popularity', fontsize=12, weight='bold')
axes[0, 1].set_xlabel('Tag Index (Sorted by Global Popularity)')
axes[0, 1].set_ylabel('Number of Albums Sharing Tag (Log Scale)')

# --- Album Labels ---
sns.histplot(labels_per_album, bins=range(0, 10), ax=axes[1, 0], color='#E056FD', kde=False)
axes[1, 0].set_title('Album Labels: Profile Complexity (Labels per Album)', fontsize=12, weight='bold')
axes[1, 0].set_xlabel('Number of Unique Record Labels on a Single Album')
axes[1, 0].set_ylabel('Count of Albums')
axes[1, 0].set_xlim(0, 6)

# album_label_popularity pre-computed in load cell (albums per label, sorted desc)
axes[1, 1].plot(album_label_popularity, color='#E056FD', linewidth=2.5)
axes[1, 1].fill_between(range(len(album_label_popularity)), album_label_popularity, color='#E056FD', alpha=0.25)
axes[1, 1].set_yscale('log')
axes[1, 1].set_title('Album Labels: Long-Tail Feature Popularity', fontsize=12, weight='bold')
axes[1, 1].set_xlabel('Label Index (Sorted by Global Popularity)')
axes[1, 1].set_ylabel('Number of Albums Sharing Label (Log Scale)')

# --- Artist Tags ---
sns.histplot(tags_per_artist, bins=range(0, 45), ax=axes[2, 0], color='#10AC84', kde=True)
axes[2, 0].set_title('Artist Tags: Profile Complexity (Tags per Artist)', fontsize=12, weight='bold')
axes[2, 0].set_xlabel('Number of Unique Tags on a Single Artist')
axes[2, 0].set_ylabel('Count of Artists')
axes[2, 0].set_xlim(0, 40)

# artist_tag_popularity pre-computed in load cell (artists per tag, sorted desc)
axes[2, 1].plot(artist_tag_popularity, color='#10AC84', linewidth=2.5)
axes[2, 1].fill_between(range(len(artist_tag_popularity)), artist_tag_popularity, color='#10AC84', alpha=0.25)
axes[2, 1].set_yscale('log')
axes[2, 1].set_title('Artist Tags: Long-Tail Feature Popularity', fontsize=12, weight='bold')
axes[2, 1].set_xlabel('Tag Index (Sorted by Global Popularity)')
axes[2, 1].set_ylabel('Number of Artists Sharing Tag (Log Scale)')

plt.tight_layout()
plt.savefig('../3-features/feature_charts/sparse_features_structural_analysis.png', dpi=300)
plt.show()

### Classic power law — a few tags cover everything, most cover almost nothing

- **Album tags** are dominated by the zero bin; among tagged albums the median is ~2–3, p95 is 5.
  The top tag covers ~268k albums and only 838 of 2,684 tags reach 100+ albums.
- **Labels** are sparser still — most labelled albums carry exactly one label.
- **Artist tags** are the most skewed: 91% of artists have none.

> **Design decision — frequency threshold at 10:** keeps a community-agreement minimum. It drops a
> long tail of one-off annotations while preserving the niche-genre tags that distinguish otherwise
> similar albums. The threshold is occurrence-based, so the vocabulary grows with the data rather
> than being capped.

# Part 2 — The Label Problem

Label tags are tempting: a label's catalogue often shares a genre. But a label aggregates tags
across *every* artist it works with, so a large, diverse label's tags describe none of its albums
in particular. This part measures which labels are genuinely genre-coherent.

## Measuring label genre coherence

For each label with tags, we compute a **per-album overlap rate**: the fraction of its albums that
share at least one tag with the label's own tag vocabulary. High overlap means the label's tags
reliably predict its albums' genres. Labels at or above 60% form the allowlist used later to rescue
zero-signal albums.

In [ ]:
# album_tags_raw, album_label_raw loaded in load cell
# LABEL_OVERLAP_THRESHOLD defined in imports

# Labels that carry tag data
label_tag_rows   = album_label_raw[album_label_raw['tag_count'] > 0][['label_id', 'tag_id']].drop_duplicates()
labels_with_tags = set(label_tag_rows['label_id'].unique())

al_deduped  = album_label_raw.drop_duplicates(subset=['album_id', 'label_id'])[['label_id', 'album_id']]
al_relevant = al_deduped[al_deduped['label_id'].isin(labels_with_tags)]

albums_with_direct_tags = set(album_tags_raw['album_id'].unique())
al_comparable = al_relevant[al_relevant['album_id'].isin(albums_with_direct_tags)]
total_per_label = al_comparable.groupby('label_id')['album_id'].nunique()

overlap_check = (
    al_comparable
    .merge(label_tag_rows, on='label_id', how='inner')
    .merge(album_tags_raw[['album_id', 'tag_id']].drop_duplicates(), on=['album_id', 'tag_id'], how='inner')
    [['label_id', 'album_id']]
    .drop_duplicates()
)
overlapping_per_label = overlap_check.groupby('label_id')['album_id'].nunique()

per_label_overlap = (
    pd.DataFrame({'total': total_per_label, 'overlapping': overlapping_per_label})
    .fillna(0)
    .assign(overlap_rate=lambda d: d['overlapping'] / d['total'])
)

label_allowlist = set(per_label_overlap[per_label_overlap['overlap_rate'] >= LABEL_OVERLAP_THRESHOLD].index)

print(f'Labels assessed                   : {len(per_label_overlap):,}')
print(f'Allowlist (>={LABEL_OVERLAP_THRESHOLD*100:.0f}% overlap)        : {len(label_allowlist):,}  ({len(label_allowlist)/len(per_label_overlap)*100:.1f}%)')
print(f'Excluded from allowlist           : {len(per_label_overlap) - len(label_allowlist):,}')

### Only a minority of tagged labels are genre-coherent enough to trust

The allowlist captures the labels whose tags actually describe their albums. The majority are
excluded — their tags are catalogue-wide averages, not per-album genre signals.

## Does label size predict coherence?

The intuitive filter would be "trust small boutique labels, distrust big ones." This checks whether
that intuition holds by plotting overlap rate across label-size buckets.

In [ ]:
label_sizes = al_deduped.groupby('label_id')['album_id'].nunique().reset_index(name='album_count')
plot_df = per_label_overlap.merge(label_sizes, on='label_id')

bins   = [0, 10, 50, 200, 1000, 10_000_000]
labels_b = ['1–10', '11–50', '51–200', '201–1000', '1000+']
plot_df['size_bin'] = pd.cut(plot_df['album_count'], bins=bins, labels=labels_b)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: boxplot of overlap rate by size bin
plot_df.boxplot(column='overlap_rate', by='size_bin', ax=axes[0], grid=False)
axes[0].set_title('Overlap Rate by Label Size', weight='bold')
axes[0].set_xlabel('Albums per Label')
axes[0].set_ylabel('Per-Album Overlap Rate')
axes[0].axhline(LABEL_OVERLAP_THRESHOLD, color='red', linestyle='--', label=f'{LABEL_OVERLAP_THRESHOLD*100:.0f}% threshold')
axes[0].legend()
plt.sca(axes[0])
plt.title('')
plt.suptitle('')

# Right: allowlist pass rate by size bin
pass_rates = (
    plot_df.groupby('size_bin', observed=True)
    .apply(lambda g: (g['overlap_rate'] >= LABEL_OVERLAP_THRESHOLD).mean() * 100)
    .reset_index(name='pct_qualifying')
)
axes[1].bar(pass_rates['size_bin'].astype(str), pass_rates['pct_qualifying'], color='#3498DB')
axes[1].set_title('% of Labels Qualifying for Allowlist by Size', weight='bold')
axes[1].set_xlabel('Albums per Label')
axes[1].set_ylabel('% of Labels with ≥60% Overlap')
axes[1].axhline(50, color='grey', linestyle=':', linewidth=1)

plt.tight_layout()
plt.show()

print('\nAllowlist qualification rate by size bin:')
for _, row in pass_rates.iterrows():
    grp = plot_df[plot_df['size_bin'] == row['size_bin']]
    n_total = len(grp)
    n_pass  = (grp['overlap_rate'] >= LABEL_OVERLAP_THRESHOLD).sum()
    print(f"  {str(row['size_bin']):>10} albums: {n_pass:>5,} / {n_total:>5,} qualify  ({row['pct_qualifying']:.1f}%)")

### Size is a poor proxy — coherence is bimodal at every size

Small labels do *not* reliably have high overlap, and some very large labels (specialist subsidiaries
with one consistent tag) hit 100%. Overlap rate is bimodal across all size bins: many labels sit at
0% (pure noise) and many at 100% (clean signal).

> **Design decision — allowlist on overlap rate, not label size or type:** size and `label_type`
> correlate too weakly with what we care about. The overlap rate measures genre coherence directly,
> is computed once at build time, and is stored as a set of label IDs.

# Part 3 — Building the Blend

The genre matrix combines the sources in three tiers:
1. **Album tags** (w=1.0) and **artist tags** (w=0.5) — applied to all albums
2. **Label reinforcement** (w=0.3) — label tags *masked* to only boost tags an album/artist already has
3. **Label rescue** (w=0.3) — allowlist label tags allowed to introduce signal, but only for albums
   with no other signal

This part validates tiers 2 and 3.

## Tier 2 — Label reinforcement via masking

Label tags are kept only where the album or artist already has signal for that tag. A Columbia
"jazz" tag boosts an album already tagged jazz; Columbia's "pop"/"rock" tags are zeroed for that
album. The cells below rebuild the raw and masked label blocks to measure how much is filtered.

In [ ]:
# All parquets, genre_tag_index, and album_artists already loaded in the load cell.
# Prepare the label tag slice used by the masking analysis below.
label_tags_raw = (
    album_label_raw[album_label_raw['tag_count'] > 0][['album_id', 'tag_id', 'tag_count']]
    .copy()
)
print(f'Label tag entries (tag_count > 0): {len(label_tags_raw):,}')

In [ ]:
# ── Raw label block (no masking) ──────────────────────────────────────────
lt = label_tags_raw[label_tags_raw['tag_id'].isin(genre_tag_index)].copy()
lt_totals   = lt.groupby('album_id')['tag_count'].transform('sum')
lt['weight'] = (lt['tag_count'] / lt_totals * W_LABEL).astype('float32')

row_idx = album_index.get_indexer(lt['album_id'].values)
col_idx = genre_tag_index.get_indexer(lt['tag_id'].values)
valid   = (row_idx >= 0) & (col_idx >= 0)

X_label_raw = csr_matrix(
    (lt['weight'].values[valid], (row_idx[valid], col_idx[valid])),
    shape=(n_albums, n_tags),
)

# ── Signal mask: album + artist tags combined (matches production pipeline) ─
# X_album_in_genre_vocab already built in load cell.
# Build artist-tags-on-albums in genre vocab space (binary).
_art = artist_tags_on_albums[artist_tags_on_albums['tag_id'].isin(genre_tag_index)]
_art_row = album_index.get_indexer(_art['album_id'].values)
_art_col = genre_tag_index.get_indexer(_art['tag_id'].values)
_art_ok  = (_art_row >= 0) & (_art_col >= 0)
X_artist_in_genre_vocab = csr_matrix(
    (np.ones(_art_ok.sum(), dtype='float32'), (_art_row[_art_ok], _art_col[_art_ok])),
    shape=(n_albums, n_tags),
)

signal_mask  = (X_album_in_genre_vocab + X_artist_in_genre_vocab) > 0
X_label_masked = X_label_raw.multiply(signal_mask)

kept    = X_label_masked.nnz
raw     = X_label_raw.nnz
dropped = raw - kept

print(f'Raw label tag entries   : {raw:>10,}')
print(f'Kept after masking      : {kept:>10,}  ({kept/raw*100:.1f}%)')
print(f'Masked out (noise)      : {dropped:>10,}  ({dropped/raw*100:.1f}%)')

### The majority of raw label-tag entries are genre-irrelevant to their albums

The masking removes most label-tag entries — confirming that, applied unconditionally, label tags
would have been mostly noise. What survives is the minority that reinforces genre signal already
present on the album or its primary artist. The mask uses the combined album + artist signal as its
basis, which matches the production pipeline in `02-feature-genre.ipynb`.

In [ ]:
# Per-album: how many label tags did each album have before and after masking?
label_raw_per_album    = np.diff(X_label_raw.indptr)
label_masked_per_album = np.diff(X_label_masked.indptr)

# Only consider albums that had any label tags at all
has_label = label_raw_per_album > 0

raw_for_labelled    = label_raw_per_album[has_label]
masked_for_labelled = label_masked_per_album[has_label]
overlap_rate        = masked_for_labelled / raw_for_labelled

print(f'Albums with any label tags: {has_label.sum():,}')
print()
print('Overlap rate (fraction of label tags that pass the mask):')
for pct in [0, 0.25, 0.5, 0.75, 1.0]:
    c = (overlap_rate <= pct).sum()
    print(f'  <= {pct*100:>3.0f}% overlap: {c:>10,}  ({c/len(overlap_rate)*100:.1f}% of labelled albums)')
print()
print(f'Mean overlap rate : {overlap_rate.mean()*100:.1f}%')
print(f'Median overlap rate: {np.median(overlap_rate)*100:.1f}%')
print(f'Albums where 0% of label tags pass (all masked out): {(overlap_rate == 0).sum():,}  ({(overlap_rate==0).mean()*100:.1f}%)')
print(f'Albums where 100% of label tags pass (full overlap): {(overlap_rate == 1.0).sum():,}  ({(overlap_rate==1.0).mean()*100:.1f}%)')

### The split is clean: most albums get all label tags dropped, a coherent minority keep them all

The per-album overlap distribution is bimodal — albums are mostly either 0% (label tags entirely
masked, the diverse-label case) or 100% (specialist label whose tags match the album or artist).
This is the intended behaviour: the mask cleanly separates noise from reinforcement.

In [ ]:
# Does masking change overall genre matrix coverage?
# Compare genre signal with unmasked vs masked label tags.
# Use X_album_in_genre_vocab (built in the previous cell) — same vocabulary as the label blocks.
genre_unmasked_approx = X_album_in_genre_vocab + X_label_raw
genre_masked_approx   = X_album_in_genre_vocab + X_label_masked

coverage_album_only = (np.diff(X_album_in_genre_vocab.indptr) > 0).sum()
coverage_unmasked   = (np.diff(genre_unmasked_approx.indptr) > 0).sum()
coverage_masked     = (np.diff(genre_masked_approx.indptr) > 0).sum()

print(f'Coverage — album tags only     : {coverage_album_only:>10,}  ({coverage_album_only/n_albums*100:.1f}%)')
print(f'Coverage — with unmasked labels: {coverage_unmasked:>10,}  ({coverage_unmasked/n_albums*100:.1f}%)')
print(f'Coverage — with masked labels  : {coverage_masked:>10,}  ({coverage_masked/n_albums*100:.1f}%)')
print(f'\nCoverage lost to masking: {coverage_unmasked - coverage_masked:>+,} albums')
print('(albums where label tags were the only source of genre signal — pure label noise)')


### Masking correctly forfeits ~252k albums of false coverage

Those are albums whose *only* genre signal came from label tags that didn't match anything on the
album. Under the old unconditional approach they received a genre fingerprint derived from the
label's whole catalogue. Dropping them to zero is the right call — better no genre opinion than a
wrong one. Tier 3 then selectively rescues the subset of these where the label is genuinely coherent.

> **Design decision — accept the coverage loss:** false coverage is worse than no coverage. The
> forfeited albums fall back on label identity, ratings, country, and era for recommendations.

# Part 4 — Coverage vs Richness

This is the part that explains a counterintuitive result. We made two changes — artist tags applied
universally, and an allowlist rescue for zero-signal albums — and measured each one's effect on
coverage in isolation.

In [ ]:
# All parquets, genre_tag_index, album_artists, artist_tags_on_albums, and
# X_album_in_genre_vocab are loaded/built in the load cell.

# ── Artist tag block — all albums (new "universal" behaviour) ──────────────
art_all = (
    artist_tags_on_albums[artist_tags_on_albums['tag_id'].isin(genre_tag_index)]
    .copy()
)
art_totals      = art_all.groupby('album_id')['tag_count'].transform('sum')
art_all['weight'] = (art_all['tag_count'] / art_totals * 0.5).astype('float32')
row_idx = album_index.get_indexer(art_all['album_id'].values)
col_idx = genre_tag_index.get_indexer(art_all['tag_id'].values)
valid   = (row_idx >= 0) & (col_idx >= 0)
X_artist_all = csr_matrix(
    (art_all['weight'].values[valid], (row_idx[valid], col_idx[valid])),
    shape=(n_albums, n_tags),
)

# ── Artist tag block — sparse albums only (old behaviour: < 5 direct tags) ─
# tags_per_album is pre-computed in load cell from mb_album_tag.parquet
sparse_ids  = album_index[tags_per_album < 5]
art_sp      = art_all[art_all['album_id'].isin(sparse_ids)]
art_sp_totals = art_sp.groupby('album_id')['tag_count'].transform('sum')
art_sp = art_sp.copy()
art_sp['weight'] = (art_sp['tag_count'] / art_sp_totals * 0.5).astype('float32')
row_idx = album_index.get_indexer(art_sp['album_id'].values)
col_idx = genre_tag_index.get_indexer(art_sp['tag_id'].values)
valid   = (row_idx >= 0) & (col_idx >= 0)
X_artist_sparse = csr_matrix(
    (art_sp['weight'].values[valid], (row_idx[valid], col_idx[valid])),
    shape=(n_albums, n_tags),
)

# ── Coverage comparison ────────────────────────────────────────────────────
# X_album_in_genre_vocab (binary album tags, genre vocab) reused from load cell
baseline_base  = X_album_in_genre_vocab + X_artist_sparse
universal_base = X_album_in_genre_vocab + X_artist_all

def cov(mat):
    return (np.diff(mat.indptr) > 0).sum()

c_baseline  = cov(baseline_base)
c_universal = cov(universal_base)
c_final     = cov(X_genre)

print(f'Coverage comparison (n={n_albums:,}):')
print()
print(f'  1. Baseline (sparse artist blend)    : {c_baseline:>10,}  ({c_baseline/n_albums*100:.1f}%)')
print(f'  2. Artist universal                  : {c_universal:>10,}  ({c_universal/n_albums*100:.1f}%)  +{c_universal-c_baseline:,}')
print(f'  3. + allowlist zero-signal labels    : {c_final:>10,}  ({c_final/n_albums*100:.1f}%)  +{c_final-c_universal:,}')

### Universal artist tags add **+0 coverage** — and that is correct, not a bug

The "+0" is provable, not a coincidence. Coverage asks whether an album has *any* signal. Compare
the old rule (artist tags only for albums with < 5 direct tags) to the new rule (artist tags for all):

| Direct album tags | Old: sparse blend | New: universal | Coverage effect |
|---|---|---|---|
| 0 tags | gets artist tags | gets artist tags | identical |
| 1–4 tags | already covered by album tags | already covered | identical |
| ≥5 tags | **excluded** from artist blend | **gets** artist blend | already covered → **identical** |

The only albums that *newly* receive artist tags under the universal rule are the ≥5-direct-tag
albums — which were already covered by their own tags. So coverage cannot move.

**What universal artist tags actually buy is richness, not coverage.** A well-tagged rock album now
also carries its artist's broader genre profile, sharpening cosine similarity. That gain is invisible
to a coverage count — it shows up in the richness distribution below.

The allowlist rescue, by contrast, *does* add genuine new coverage (~+13k albums) — albums that had
zero signal from their own tags and artist, where a genre-coherent label is the only available
indication.

In [ ]:
# Coverage vs richness — two different things the blend does.
# Coverage  = does an album have ANY signal?  (binary)
# Richness  = how many distinct genre tags does an already-covered album carry?

had_direct = tags_per_album > 0
before = tags_per_album[had_direct]      # direct album tags only
after  = genre_per_album[had_direct]     # final genre matrix (album + artist + label)
gain   = after - before

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: distribution before vs after, for albums that already had direct tags
sns.histplot(before, bins=range(0, 25), ax=axes[0], color='#4A90E2', label='Direct album tags', alpha=0.55)
sns.histplot(after,  bins=range(0, 25), ax=axes[0], color='#27AE60', label='Final genre matrix', alpha=0.55)
axes[0].set_title('Tags per Album — Direct vs Blended\n(albums that already had direct tags)', weight='bold')
axes[0].set_xlabel('Distinct Tags on the Album')
axes[0].set_ylabel('Count of Albums')
axes[0].set_xlim(0, 22)
axes[0].legend()

# Right: per-album richness gain
sns.histplot(gain, bins=range(0, 25), ax=axes[1], color='#8E44AD')
axes[1].set_title('Richness Gain per Album\n(genre tags − direct tags)', weight='bold')
axes[1].set_xlabel('Additional Tags Gained from Blending')
axes[1].set_ylabel('Count of Albums')
axes[1].set_xlim(0, 22)

plt.tight_layout()
plt.show()

print(f'Albums that already had direct tags : {had_direct.sum():,}')
print(f'Median distinct tags — direct only  : {np.median(before):.0f}')
print(f'Median distinct tags — genre matrix : {np.median(after):.0f}')
print(f'Median richness gain per album      : {np.median(gain):.0f}')
print(f'Mean richness gain per album        : {gain.mean():.1f}')

### The blend enriches already-covered albums even when it can't extend coverage

Among albums that already had direct tags, the genre matrix carries a higher median tag count than
direct tags alone. This is the richness gain from universal artist + reinforced label tags — exactly
the benefit the "+0 coverage" result hides. Richer profiles mean more precise similarity, which is
the real payoff of the universal-artist change.

### ~68.8% is close to the tag-signal ceiling

After all three tiers, ~31% of albums still have no genre signal: no album tags, an untagged artist,
and no allowlist label. There is no honest tag-based signal left for them. The remaining levers each
have a cost:
- **Lower the 60% allowlist threshold** — recovers some coverage but reintroduces the label noise
  Part 2/3 removed.
- **Accept the ceiling** — these albums lean on the other feature blocks (country, era, ratings,
  label identity), which is reasonable.

> **Design decision — treat 68.8% as the ceiling rather than chase coverage with noise.** The
> 46% → 68.8% gain comes from clean, validated signal; pushing further trades quality for a coverage
> number that wouldn't improve recommendations.

# Part 5 — Column Vocabulary & Pruning

The `MIN_TAG_OCC = 10` threshold sets how many tag columns survive. This checks what it actually
cuts.

In [ ]:
# tag_col_counts and label_col_counts pre-computed in load cell from raw parquets
print('=== Album Tag Columns ===')
total_tag_mass = tag_col_counts.sum()
for threshold in [1, 10, 50, 100, 500, 1000]:
    surviving = (tag_col_counts >= threshold).sum()
    mass = tag_col_counts[tag_col_counts >= threshold].sum()
    print(f'  Threshold >= {threshold:>5}: {surviving:>6,} columns  ({mass/total_tag_mass*100:.1f}% of total tag-album pairs)')

print()
print('=== Album Label Columns ===')
total_label_mass = label_col_counts.sum()
for threshold in [1, 10, 50, 100, 500]:
    surviving = (label_col_counts >= threshold).sum()
    mass = label_col_counts[label_col_counts >= threshold].sum()
    print(f'  Threshold >= {threshold:>5}: {surviving:>6,} columns  ({mass/total_label_mass*100:.1f}% of total label-album pairs)')

### The threshold drops most columns but keeps almost all the signal mass

Tags below 10 occurrences are numerous but collectively cover very few albums, so pruning them costs
little signal while sharply reducing column count and sparsity. The label vocabulary follows the same
power law — a few majors cover hundreds of thousands of albums, most imprints only a handful.

> **Design decision — occurrence threshold, not a fixed column target:** vocabulary size stays
> data-driven and grows with tagging activity. The KNN training notebooks apply a secondary
> `safe_threshold` prune so no album loses all its columns.